<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB15_Case_Study_CLIWOC_Nationality_from_Ship_Routes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB15 · Class 15 — Case Study: CLIWOC Historical Ship Logbooks, Classifying Nationality from Routes**

## Block 4: Proyectos — Case Studies (continued)

`NB14` used real 21st-century weather data. This case study reaches back much further: real 18th-century ship logbooks, digitized for climate research, used here to ask a different kind of question — can a ship's **route and timing alone** reveal which nation it sailed for?

**The real-world problem**: between 1750 and 1799, the British, Dutch, Spanish, and French each ran established maritime trade networks tied to their colonial possessions — Spain's transatlantic and Manila galleon routes, the Dutch VOC's Cape route to the East Indies, British East India Company and Atlantic trade, French Caribbean and Indian Ocean trade. If those networks were real and geographically distinct, a model should be able to recover "which nation" from nothing but *where* and *when* a ship was — a genuine test of whether real historical trade geography is learnable from data, not an arbitrary classroom label.

Following `NB14`'s corrected structure, this class again has **two parts**: **Part A** (Sections 5–8) validates a modeling approach with a standard random split; **Part B** (Sections 9–10) is the real test — training only on earlier decades (1750–1789) and predicting nationality for a **later decade the model has never seen** (1790–1799), checking whether these trade-route patterns actually held stable across time, or shifted.

### Learning objectives

By the end of this class, students will be able to:
- Explain why "route → nationality" is a real, historically grounded question, not an arbitrary label.
- Work with a real, messy historical dataset, including realistic class imbalance.
- Visualize class-separated geographic data on a real map and use it to sanity-check a modeling premise before training anything.
- Handle class imbalance with a naive baseline and `class_weight="balanced"`.
- Design a temporal holdout that matches the real question being asked (generalizing across decades, not being told the "answer" it already saw).

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | What is CLIWOC, and why "route → nationality" is a real question | 15 min | Theory |
| 3 | Loading the real, curated dataset | 10 min | Practice |
| 4 | Exploring the data: routes, nationalities, time span | 20 min | Practice |
| 5 | Preparing features and confronting real class imbalance | 10 min | Theory + Practice |
| 6 | Applying `NB13`'s decision framework | 5 min | Theory + Practice |
| 7 | Part A: training and comparing models (methodology validation) | 15 min | Practice |
| 8 | Part A: evaluation | 10 min | Practice |
| 9 | Part B: a genuine test — forecasting a held-out decade | 15 min | Practice |
| 10 | Interpreting Part B, and connecting it to real history | 10 min | Practice |
| 11 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB14`**: real ECMWF weather data, a genuine held-out-year forecast, and the lesson that a fair generalization test has to hold out the *right* variable (a year, not a season, when the target is seasonal).
- **`NB15`** (today): a different kind of real data — historical, sparse, imbalanced — and the same discipline applied to a temporal holdout that matches *this* problem's real question.

---

## 2. What is CLIWOC, and why "route → nationality" is a real question

**[CLIWOC](https://en.wikipedia.org/wiki/CLIWOC)** (Climatological Database for the World's Oceans) was a real research project that converted historical ships' logbooks — British, Dutch, French, and Spanish, 1750–1850 — into a standardized digital database, originally to reconstruct historical climate and wind patterns from centuries of daily noon observations. That means every row in this dataset is a **real entry a real ship's officer wrote down** at sea, up to 275 years ago.

Today's question uses the same data for a different purpose: each of the four nations ran distinct, real trade networks shaped by their colonial territories and monopolies — Spain's transatlantic and Pacific galleon routes, the Dutch East India Company's Cape-of-Good-Hope route to Indonesia, British Atlantic and Indian Ocean trade, French Caribbean and Indian Ocean trade. If those networks were geographically real and distinct (which real maritime history says they were), a model trained only on **where** and **when** a logbook entry was recorded should be able to recover **which nation** wrote it — genuine historical geography, learnable from data, not an arbitrary label invented for a homework problem.

---

## 3. Loading the real, curated dataset

The full CLIWOC database has over 287,000 real logbook entries and 180 columns spanning 1662–1855. For a focused 2-hour class, we use a curated slice: entries from **1750–1799** (the best-covered, most balanced period across all four nations), keeping only the columns needed for this task:

In [ ]:
!wget -q -O cliwoc.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/CLIWOC_curated_1750_1799.csv

import pandas as pd

cliwoc = pd.read_csv("cliwoc.csv")
print(cliwoc.shape)
cliwoc.head()

---

## 4. Exploring the data: routes, nationalities, time span

`NB02`'s starting questions, on real 18th-century data this time:

In [ ]:
print(cliwoc["Nationality"].value_counts())
print()
print("Year range:", cliwoc["Year"].min(), "-", cliwoc["Year"].max())

**Read your own output**: the four classes are not close to balanced — French entries are noticeably rarer than the other three. This reflects real archive coverage (how many French logbooks survived and were digitized), not a genuine historical absence of French ships; it is exactly the kind of real-world imbalance `NB09` flagged and worth carrying forward here.

Before training anything, test the actual premise visually: does each nation's real logbook data trace a geographically distinct pattern?

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sample = cliwoc.sample(5000, random_state=42)
colors = {"BRITISH": "tab:blue", "DUTCH": "tab:orange", "SPANISH": "tab:green", "FRENCH": "tab:red"}

fig = plt.figure(figsize=(12, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="whitesmoke")

for nation, color in colors.items():
    subset = sample[sample["Nationality"] == nation]
    ax.scatter(subset["longitude"], subset["latitude"], s=4, alpha=0.5,
               label=nation, color=color, transform=ccrs.PlateCarree())

ax.legend(markerscale=3, loc="lower left")
ax.set_title("Real CLIWOC logbook positions by nationality, 1750-1799 (5,000-entry sample)")
plt.show()

**Read your own map**: can you see the Spanish transatlantic/Pacific routes, the Dutch Cape-of-Good-Hope corridor toward Indonesia, the British Atlantic and Indian Ocean presence? If these four colors separate into visually distinct regions, that is real, direct evidence the "route reveals nationality" premise holds *before* we ever train a model — the map itself is doing genuine exploratory data analysis, `NB02`-style, on a real historical question.

---

## 5. Preparing features and confronting real class imbalance

Features: `latitude`, `longitude` (where), `Year`, `Month` (when — `Month` can matter too, since some routes were timed around trade winds or monsoon seasons). Target: `Nationality`.

In [ ]:
feature_cols = ["latitude", "longitude", "Year", "Month"]
X = cliwoc[feature_cols]
y = cliwoc["Nationality"]

print(y.value_counts(normalize=True).round(3))

With French at roughly 5% of the data, a model that ignores French entirely could still score high overall accuracy — `NB03`'s original imbalance warning, now with real numbers behind it. We'll use two concrete tools against this: a **naive baseline** to know what "cheating by ignoring the minority class" would actually score, and scikit-learn's `class_weight="balanced"` option, which up-weights minority-class errors during training instead of treating every mistake equally.

---

## 6. Applying `NB13`'s decision framework

1. **Labels?** Yes — `Nationality` is real and known for every entry.
2. **Data shape?** Tabular — four simple numeric features per row.
3. **Data volume?** ~150,000 rows — the largest dataset used in this course so far, comfortably in the range where either classical ML or a neural network could work; `NB13`'s framework doesn't strongly favor one here the way it did for `NB06`'s 308-row yacht data.

Given the framework doesn't push hard in either direction this time, we'll use a classical ensemble (fast, interpretable, easy to weight for imbalance) — but note for yourself that this is a case where trying a small MLP (`NB07` style) as homework is a genuinely open question, not a foregone conclusion.

---

## 7. Part A: training and comparing models (methodology validation)

A random, stratified split first — the same kind of "validate the approach" step as `NB14`'s Part A:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

Compare the naive baseline against a class-weighted Random Forest:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)
print("Naive baseline (always predict the majority class) accuracy:", round(baseline.score(X_test_scaled, y_test), 3))

rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf.fit(X_train_scaled, y_train)
print("Random Forest accuracy:", round(rf.score(X_test_scaled, y_test), 3))

---

## 8. Part A: evaluation

Overall accuracy hides how each individual nation is doing — a full report matters more here than usual, given the imbalance:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = rf.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred, labels=rf.classes_))
print()
print(classification_report(y_test, y_pred, labels=rf.classes_))

**Read your own report**: is French recall noticeably lower than the other three nations, even with `class_weight="balanced"`? That would be an honest, expected finding given how little French data exists — a real limitation of the underlying archive, not a modeling mistake to fix away.

---

## 9. Part B: a genuine test — forecasting a held-out decade

Part A's random split mixes entries from every decade into both train and test — a fair methodology check, but not a real test of whether these route patterns *held up over time*. The genuine question: train only on **1750–1789**, then predict nationality for **1790–1799 — a full decade the model has never seen** — exactly the same discipline `NB14` used for its held-out year, applied to the variable that actually matters for *this* question (time period, not season):

In [ ]:
train_era = cliwoc[cliwoc["Year"] <= 1789]
test_era = cliwoc[cliwoc["Year"] >= 1790]

X_train_era, y_train_era = train_era[feature_cols], train_era["Nationality"]
X_test_era, y_test_era = test_era[feature_cols], test_era["Nationality"]

print("Train (1750-1789):", X_train_era.shape)
print("Test  (1790-1799):", X_test_era.shape)
print(y_test_era.value_counts(normalize=True).round(3))

Train a fresh model — this must **not** reuse the Part A model, which already saw some 1790s rows in its own training split:

In [ ]:
scaler_era = StandardScaler()
X_train_era_scaled = scaler_era.fit_transform(X_train_era)
X_test_era_scaled = scaler_era.transform(X_test_era)

rf_era = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf_era.fit(X_train_era_scaled, y_train_era)

y_pred_era = rf_era.predict(X_test_era_scaled)
print(classification_report(y_test_era, y_pred_era, labels=rf_era.classes_))

---

## 10. Interpreting Part B, and connecting it to real history

**Compare this report to Part A's.** A meaningful drop here would be a genuinely interesting historical finding, not a failure: it would suggest 1790s trade routes shifted from the 1750–1789 pattern the model learned. That decade includes real, disruptive events worth knowing about before over-interpreting any single number — the American Revolutionary War (1775–1783) reshaped British Atlantic trade, and the French Revolution began in 1789, right at the training/test boundary, with major disruptions to French shipping following soon after.

This is exactly the honest use of a genuine holdout: it doesn't just score a model, it can **surface a real historical question worth investigating further** (was there a real shift, and if so, in which nation's routes specifically?) — a hypothesis this notebook raises but does not claim to prove; you would need real historical analysis, not just one accuracy number, to confirm it.

---

## Class summary

- CLIWOC turns 18th-century ship logbooks into real, structured data — real observations, real class imbalance, real historical stakes behind the numbers.
- "Route reveals nationality" is a genuine historical claim, checked visually on a real map before any model touched the data.
- A naive baseline and `class_weight="balanced"` are two concrete, complementary tools against real class imbalance — used together, not as a substitute for reading the full per-class report.
- Part A (random split) validates a modeling approach; Part B (train on early decades, test on a later held-out decade) is the real test — matching the holdout variable to the actual question, exactly as `NB14` established.
- A real accuracy drop across a genuine holdout isn't a failure to explain away — it's a legitimate signal worth connecting to real history.

## For the next class

Another Block 4 case study, using real terrain/elevation data — continuing the same pattern: a real dataset, a real question, and a genuine (not just methodological) holdout wherever the question calls for one.

## Homework / Practice Ideas

1. Add `ShipType` (from the raw dataset, encoded) as a fifth feature — does it improve Part A's per-class recall, especially for French?
2. Try a small MLP (`NB07` style) on this task, per Section 6's open question — does it beat the Random Forest here, given ~150,000 rows is far more data than most of this course's other classification tasks had?
3. Restrict Part B's test set to just 1789–1791 (around the French Revolution's start) instead of the full 1790s — does French recall specifically drop more sharply right around that boundary?
4. Recompute Part A using `class_weight=None` (the default, unweighted) instead of `"balanced"` — how much does French recall change, and does overall accuracy go up or down?
5. Using the map from Part 4, pick one nation and describe (in a markdown cell) what its real historical trade routes should look like — does the plotted data match your own historical expectation?

> ***As always: a real historical dataset can teach you as much about history as about machine learning — Part 10's interpretation is not optional decoration, it's the actual point of using data this old.***
